# UTS SOAL 3 — Spark DataFrame & SQL (25 poin)
Notebook terpisah, membaca snapshot **offset tetap** yang sama dengan Soal 2. Kernel **BigDataLab**; Master terbaru `192.168.1.32`.

### SEL 1 — konfigurasi

In [1]:
# WAJIB DIJALANKAN PERTAMA SETIAP KALI KERNEL DIRESTART. Tidak perlu config.py terpisah.
import os, sys, json, time, socket, datetime, importlib.util
from pathlib import Path
MASTER_IP = os.environ.get('BIGDATA_MASTER_IP', '192.168.1.32')  # terbaru: panel MASTER
KAFKA_BOOTSTRAP = os.environ.get('BIGDATA_KAFKA', MASTER_IP + ':9092')
KAFKA_TOPIC = os.environ.get('BIGDATA_TOPIC', 'wiki.recentchange.raw')
SPARK_MASTER = os.environ.get('BIGDATA_SPARK', 'spark://' + MASTER_IP + ':7077')
# Hasil/output disimpan pada komputer MASTER tempat kernel Jupyter berjalan.
DEFAULT_DIR = 'C:/BigData/work/UTS_Wikipedia_2026' if os.name == 'nt' else str(Path.cwd() / '_hasil_uts')
BASE = Path(os.environ.get('BIGDATA_OUTPUT_DIR', DEFAULT_DIR))
DATA, HASIL, STATE = BASE/'data', BASE/'hasil', BASE/'state'
for folder in (DATA,HASIL,STATE):folder.mkdir(parents=True,exist_ok=True)
MANIFEST_PATH = DATA/'snapshot_offsets.json'
print('KERNEL:',sys.executable)
print('HOST KERNEL:',socket.gethostname())
print('MASTER:',SPARK_MASTER,'| KAFKA:',KAFKA_BOOTSTRAP,'| TOPIC:',KAFKA_TOPIC)
print('FOLDER HASIL:',BASE)


KERNEL: C:\BigData\venv\Scripts\python.exe
HOST KERNEL: BIGDATA-MASTER
MASTER: spark://192.168.1.32:7077 | KAFKA: 192.168.1.32:9092 | TOPIC: wiki.recentchange.raw
FOLDER HASIL: C:\BigData\work\UTS_Wikipedia_2026


### SEL 2 — paket Kafka

In [2]:
# CEK PERPUSTAKAAN DAN KONEKSI. Jangan otomatis memasang ulang paket bawaan BigDataLab.
needed = {'kafka':'kafka-python','requests':'requests'}
missing = [package for module,package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError('Paket Python belum tersedia: '+', '.join(missing)+
      '. Jalankan DI KERNEL BigDataLab: %pip install '+ ' '.join(missing))
from kafka import KafkaConsumer, KafkaAdminClient, TopicPartition, KafkaProducer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
for port,label in [(7077,'Spark Master'),(9092,'Kafka')]:
    try:
        with socket.create_connection((MASTER_IP,port),timeout=4):
            print(label,port,'TCP TERJANGKAU (bukan jaminan metadata/broker valid).')
    except OSError as exc:
        print(label,port,'TCP GAGAL:',exc)
print('LIBRARY KAFKA TERSEDIA; langkah berikutnya mengecek metadata broker.')


Spark Master 7077 TCP TERJANGKAU (bukan jaminan metadata/broker valid).
Kafka 9092 TCP TERJANGKAU (bukan jaminan metadata/broker valid).
LIBRARY KAFKA TERSEDIA; langkah berikutnya mengecek metadata broker.


### SEL 3 — Spark & konektor

In [3]:
APP_NAME='UTS_S3_DataFrame_SQL'

# Spark Standalone; driver berjalan di PC Master via JupyterLab BigDataLab.
# Pastikan kernel fresh ketika mengubah IP atau mengaktifkan konektor Spark-Kafka.
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType,StructField,LongType,IntegerType,StringType,BooleanType
SPARK_KAFKA_PACKAGE = 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3'
if not pyspark.__version__.startswith('4.1.'):
    raise RuntimeError('PySpark='+pyspark.__version__+'; cek versi Spark sebelum mengubah paket konektor!')
existing=SparkSession.getActiveSession()
if existing is not None and existing.sparkContext.master != SPARK_MASTER:
    raise RuntimeError('Sudah ada SparkSession ke '+existing.sparkContext.master+
                       '. Restart KERNEL BigDataLab lalu jalankan dari sel 1; jangan ganti variabel saja.')
spark=(SparkSession.builder.master(SPARK_MASTER)
  .appName(APP_NAME)
  .config('spark.driver.host',MASTER_IP)
  .config('spark.driver.bindAddress','0.0.0.0')
  .config('spark.executor.memory','1g')
  .config('spark.cores.max','6')
  .config('spark.jars.packages', SPARK_KAFKA_PACKAGE)
  .config('spark.sql.shuffle.partitions','8')
  .config('spark.sql.session.timeZone','UTC')
  .getOrCreate())
sc=spark.sparkContext
print('SPARK:',spark.version,'| MASTER:',sc.master,'| UI:',sc.uiWebUrl)
if sc.master!=SPARK_MASTER:
    raise RuntimeError('Spark berjalan di alamat berbeda. Restart KERNEL; periksa IP Master.')


SPARK: 4.1.3 | MASTER: spark://192.168.1.32:7077 | UI: http://192.168.1.32:4041


### SEL 4 — manifest snapshot

In [4]:
# SNAPSHOT BERSAMA 2 & 3: rentang offset Kafka dibekukan SATU KALI.
# Menyimpan offset, bukan menyalin data privat ke GitHub.
from kafka import KafkaConsumer, TopicPartition
monitor = KafkaConsumer(bootstrap_servers=[KAFKA_BOOTSTRAP],group_id=None,
                        enable_auto_commit=False,request_timeout_ms=30000)
try:
    if MANIFEST_PATH.exists():
        manifest=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        print('SNAPSHOT LAMA DIPAKAI TANPA DIUBAH:',MANIFEST_PATH)
    else:
        parts=sorted(monitor.partitions_for_topic(KAFKA_TOPIC) or [])
        if not parts:raise RuntimeError('TOPIC Kafka belum ditemukan: '+KAFKA_TOPIC)
        tps=[TopicPartition(KAFKA_TOPIC,p) for p in parts]
        begins=monitor.beginning_offsets(tps);ends=monitor.end_offsets(tps)
        manifest={'topic':KAFKA_TOPIC,'master_ip_saat_dibekukan':MASTER_IP,
          'dibuat_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
          'start':{str(tp.partition):int(begins[tp]) for tp in tps},
          'end':{str(tp.partition):int(ends[tp]) for tp in tps}}
        manifest['total_offsets']=sum(manifest['end'][p]-manifest['start'][p] for p in manifest['end'])
        if manifest['total_offsets']<=0:raise RuntimeError('Topic belum berisi data untuk snapshot.')
        MANIFEST_PATH.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
        print('SNAPSHOT BARU BEKU:',MANIFEST_PATH)
    if manifest['topic']!=KAFKA_TOPIC:
        raise RuntimeError('Topic pada snapshot berbeda! Periksa '+str(MANIFEST_PATH))
    print('PESAN DLM RENTANG OFFSET:',manifest['total_offsets'])
    print('OFFSET AWAL:',manifest['start'],'OFFSET AKHIR:',manifest['end'])
    if manifest['total_offsets']<200000:
        print('UTS 2: 200.000 ATAU seluruh data terkumpul. Jelaskan jumlah aktual dalam laporan.')
finally:monitor.close()


SNAPSHOT LAMA DIPAKAI TANPA DIUBAH: C:\BigData\work\UTS_Wikipedia_2026\data\snapshot_offsets.json
PESAN DLM RENTANG OFFSET: 453974
OFFSET AWAL: {'0': 0, '1': 0, '2': 0, '3': 0} OFFSET AKHIR: {'0': 121140, '1': 108101, '2': 113414, '3': 111319}


### SEL 5 — Kafka batch

In [5]:
# Baca Kafka secara batch pada OFFSET TETAP; tidak membutuhkan C:\shared pada Worker.
# Pastikan Spark Kafka connector sudah tersedia di JAR Spark 4.1.3.
startingOffsets=json.dumps({KAFKA_TOPIC:manifest['start']})
endingOffsets=json.dumps({KAFKA_TOPIC:manifest['end']})
try:
    raw_df=(spark.read.format('kafka')
      .option('kafka.bootstrap.servers',KAFKA_BOOTSTRAP)
      .option('subscribe',KAFKA_TOPIC)
      .option('startingOffsets',startingOffsets)
      .option('endingOffsets',endingOffsets)
      .option('failOnDataLoss','true')
      .load().select(F.col('value').cast('string').alias('raw_json')))
    print('READ KAFKA BATCH OK; DATA AKAN TERBACA SAAT ACTION .count()/.collect().')
except Exception as err:
    print('KONEKTOR SPARK-KAFKA GAGAL. Restart kernel sesudah memeriksa JAR. Error asli:')
    raise


READ KAFKA BATCH OK; DATA AKAN TERBACA SAAT ACTION .count()/.collect().


### SEL 6 — 3A: skema eksplisit, timestamp, selisih_byte, temp view

In [6]:
length_schema=StructType([StructField('old',LongType()),StructField('new',LongType())])
event_schema=StructType([
 StructField('id',LongType()),StructField('type',StringType()),StructField('wiki',StringType()),
 StructField('server_name',StringType()),StructField('title',StringType()),
 StructField('namespace',IntegerType()),StructField('user',StringType()),
 StructField('bot',BooleanType()),StructField('minor',BooleanType()),
 StructField('timestamp',LongType()),StructField('length',length_schema),
 StructField('length_old',LongType()),StructField('length_new',LongType()),
 StructField('comment',StringType())])
source=(raw_df.select(F.from_json('raw_json',event_schema).alias('evt')).select('evt.*'))
df=(source.withColumn('event_time',F.timestamp_seconds(F.col('timestamp')))
 .withColumn('old_byte',F.coalesce(F.col('length.old'),F.col('length_old'),F.lit(0)))
 .withColumn('new_byte',F.coalesce(F.col('length.new'),F.col('length_new'),F.lit(0)))
 .withColumn('selisih_byte',F.col('new_byte')-F.col('old_byte')))
df.createOrReplaceTempView('wiki_events')
print('SKEMA:');df.printSchema();print('ROWS:',df.count())
print('SAMPEL TANPA FIELD USER:');df.select('wiki','type','bot','title','event_time','selisih_byte').show(5,truncate=45)


SKEMA:
root
 |-- id: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- server_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- user: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: long (nullable = true)
 |    |-- new: long (nullable = true)
 |-- length_old: long (nullable = true)
 |-- length_new: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- old_byte: long (nullable = false)
 |-- new_byte: long (nullable = false)
 |-- selisih_byte: long (nullable = false)



ROWS: 453974
SAMPEL TANPA FIELD USER:


+------------+----------+-----+---------------------------------------------+-------------------+------------+
|        wiki|      type|  bot|                                        title|         event_time|selisih_byte|
+------------+----------+-----+---------------------------------------------+-------------------+------------+
| commonswiki|      edit| true|File:House on 34th Avenue South - DPLA - 5...|2026-11-19 11:35:27|        3299|
|wikidatawiki|      edit|false|                                     Q2808026|2026-11-19 11:35:28|        1047|
|      frwiki|categorize|false|  Catégorie:Pièce de théâtre du XVIIIe siècle|2026-11-19 11:35:25|           0|
|      ttwiki|      edit| true|Эль Серито, Колония эль Серито (Акапулко д...|2026-11-19 11:35:28|         172|
|wikidatawiki|      edit|false|                                     Q1794929|2026-11-19 11:35:28|        1047|
+------------+----------+-----+---------------------------------------------+-------------------+------------+
o

### SEL 7 — 3B(i) Jumlah edit/menit pada 5 wiki terbesar

In [7]:
sql_1='''WITH top5 AS (
 SELECT wiki FROM wiki_events WHERE type='edit' GROUP BY wiki ORDER BY COUNT(*) DESC LIMIT 5
) SELECT date_trunc('minute',e.event_time) AS menit,e.wiki,COUNT(*) AS total_edit
 FROM wiki_events e JOIN top5 t ON e.wiki=t.wiki
 WHERE e.type='edit' AND e.event_time IS NOT NULL
 GROUP BY date_trunc('minute',e.event_time),e.wiki ORDER BY menit DESC,total_edit DESC
'''
res1=spark.sql(sql_1);res1.show(30,truncate=False)


+-------------------+------------+----------+
|menit              |wiki        |total_edit|
+-------------------+------------+----------+
|2026-11-19 15:34:00|wikidatawiki|64        |
|2026-11-19 15:34:00|commonswiki |25        |
|2026-11-19 15:34:00|enwiki      |9         |
|2026-11-19 15:34:00|ttwiki      |2         |
|2026-11-19 15:33:00|wikidatawiki|533       |
|2026-11-19 15:33:00|commonswiki |202       |
|2026-11-19 15:33:00|enwiki      |100       |
|2026-11-19 15:33:00|ttwiki      |25        |
|2026-11-19 15:32:00|wikidatawiki|489       |
|2026-11-19 15:32:00|commonswiki |170       |
|2026-11-19 15:32:00|enwiki      |85        |
|2026-11-19 15:32:00|ttwiki      |27        |
|2026-11-19 15:31:00|wikidatawiki|510       |
|2026-11-19 15:31:00|commonswiki |236       |
|2026-11-19 15:31:00|enwiki      |106       |
|2026-11-19 15:31:00|ttwiki      |27        |
|2026-11-19 15:30:00|wikidatawiki|499       |
|2026-11-19 15:30:00|commonswiki |248       |
|2026-11-19 15:30:00|enwiki      |

### SEL 8 — 3B(ii) Proporsi bot/manusia per wiki + pembanding hasil RDD 2(i)

In [10]:
# SOAL 3B(ii) — Persentase bot dan manusia per wiki

sql_2 = """
SELECT
    wiki,
    COUNT(*) AS semua_event,
    SUM(CASE WHEN bot = true THEN 1 ELSE 0 END) AS bot_events,
    SUM(CASE WHEN bot = false THEN 1 ELSE 0 END) AS human_events,
    SUM(CASE WHEN bot IS NULL THEN 1 ELSE 0 END) AS bot_unknown,
    ROUND(
        100.0 * SUM(CASE WHEN bot = true THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_bot,
    ROUND(
        100.0 * SUM(CASE WHEN bot = false THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_human
FROM wiki_events
WHERE wiki IS NOT NULL
GROUP BY wiki
ORDER BY semua_event DESC
"""

res2 = spark.sql(sql_2)

print("=== PERSENTASE BOT DAN MANUSIA PER WIKI ===")
res2.show(20, truncate=False)


# Hasil SQL dengan filter yang sama seperti Soal 2B(i)
print("=== TOP 10 HUMAN EDIT UNTUK DIBANDINGKAN DENGAN SOAL 2 ===")

compare_sql = spark.sql("""
SELECT
    wiki,
    COUNT(*) AS human_edit
FROM wiki_events
WHERE type = 'edit'
  AND bot = false
  AND wiki IS NOT NULL
GROUP BY wiki
ORDER BY human_edit DESC
LIMIT 10
""")

compare_sql.show(10, truncate=False)

print("Analisis SQL selesai.")
print("Validasi dengan hasil RDD asli Soal 2 masih perlu dilakukan.")


=== PERSENTASE BOT DAN MANUSIA PER WIKI ===


+------------+-----------+----------+------------+-----------+-------+---------+
|wiki        |semua_event|bot_events|human_events|bot_unknown|pct_bot|pct_human|
+------------+-----------+----------+------------+-----------+-------+---------+
|commonswiki |137925     |56729     |81196       |0          |41.13  |58.87    |
|wikidatawiki|83767      |666       |83101       |0          |0.80   |99.20    |
|enwiki      |55843      |2285      |53558       |0          |4.09   |95.91    |
|zhwikisource|24023      |23855     |168         |0          |99.30  |0.70     |
|elwiktionary|13112      |13041     |71          |0          |99.46  |0.54     |
|mgwiktionary|11766      |11766     |0           |0          |100.00 |0.00     |
|frwiki      |8682       |579       |8103        |0          |6.67   |93.33    |
|arwiki      |7594       |5726      |1868        |0          |75.40  |24.60    |
|ruwiktionary|6555       |5752      |803         |0          |87.75  |12.25    |
|dewiki      |5869       |48

+------------+----------+
|wiki        |human_edit|
+------------+----------+
|wikidatawiki|80784     |
|enwiki      |17672     |
|commonswiki |17174     |
|frwiki      |3863      |
|dewiki      |3719      |
|ruwiki      |2163      |
|itwiki      |2082      |
|jawiki      |1786      |
|eswiki      |1464      |
|zhwiki      |1343      |
+------------+----------+

Analisis SQL selesai.
Validasi dengan hasil RDD asli Soal 2 masih perlu dilakukan.


### SEL 9 — 3B(iii) Average dan median abs(selisih_byte) per type

In [11]:
spark.sql('''SELECT type,COUNT(*) AS jumlah,
 ROUND(AVG(ABS(selisih_byte)),2) AS avg_abs_byte,
 percentile_approx(ABS(selisih_byte),0.5) AS median_abs_byte
 FROM wiki_events WHERE type IS NOT NULL GROUP BY type ORDER BY jumlah DESC
''').show(30,truncate=False)


+----------+------+------------+---------------+
|type      |jumlah|avg_abs_byte|median_abs_byte|
+----------+------+------------+---------------+
|edit      |229806|990.05      |266            |
|categorize|170872|0.0         |0              |
|log       |29216 |0.0         |0              |
|new       |24074 |2058.57     |1967           |
+----------+------+------------+---------------+



### SEL 10 — 3B(iv) Top 10 halaman idwiki dengan editor unik terbanyak

In [12]:
spark.sql('''SELECT title,COUNT(DISTINCT user) AS editor_unik,COUNT(*) AS total_edit
 FROM wiki_events WHERE wiki='idwiki' AND type='edit' AND title IS NOT NULL AND user IS NOT NULL
 GROUP BY title ORDER BY editor_unik DESC,total_edit DESC LIMIT 10
''').show(10,truncate=False)


+-------------------------------------------------+-----------+----------+
|title                                            |editor_unik|total_edit|
+-------------------------------------------------+-----------+----------+
|D'Academy (musim 8)                              |3          |12        |
|Wikipedia:Intervensi pengurus terhadap vandalisme|3          |6         |
|Ujang Darwis                                     |2          |4         |
|I Komang Koheri                                  |2          |3         |
|Firman Santyabudi                                |2          |3         |
|Sitti Umaria Suruwaky                            |2          |3         |
|Pangkalan, Pangkalan Koto Baru, Lima Puluh Kota  |2          |2         |
|Kabupaten Gianyar                                |2          |2         |
|Putri Srinual dari Lampang                       |2          |2         |
|Arisan (acara televisi)                          |2          |2         |
+------------------------

### SEL 11 — 3C 

In [13]:
import math
windowed=(df.filter((F.col('type')=='edit') & F.col('wiki').isNotNull() &
        F.col('title').isNotNull() & F.col('event_time').isNotNull())
  .groupBy('wiki','title',F.window('event_time','5 minutes').alias('win'))
  .agg(F.count('*').alias('jumlah_edit')))
if not windowed.take(1):
    print('Tidak ada edit timestamp valid untuk hitung threshold.')
else:
    qs=windowed.approxQuantile('jumlah_edit',[0.5,0.9,0.95,0.99],0.01)
    k=max(1,math.ceil(qs[-1]))
    print('P50 P90 P95 P99:',qs,'| k=ceil(P99):',k)
    spikes=(windowed.filter(F.col('jumlah_edit')>=k)
       .select('wiki','title',F.col('win.start').alias('window_start'),
               F.col('win.end').alias('window_end'),'jumlah_edit')
       .orderBy(F.desc('jumlah_edit')))
    spikes.show(30,truncate=False)
    (HASIL/'s3_spike_threshold.json').write_text(json.dumps({'quantiles':qs,'threshold_k':k},indent=2))
    print('PERHATIAN: threshold kecil (mis. k=1) dapat menghasilkan banyak false positive; diskusikan distribusi.')


P50 P90 P95 P99: [1.0, 1.0, 2.0, 59.0] | k=ceil(P99): 59


+------------+---------+-------------------+-------------------+-----------+
|wiki        |title    |window_start       |window_end         |jumlah_edit|
+------------+---------+-------------------+-------------------+-----------+
|wikidatawiki|Q64596929|2026-11-19 12:35:00|2026-11-19 12:40:00|59         |
+------------+---------+-------------------+-------------------+-----------+

PERHATIAN: threshold kecil (mis. k=1) dapat menghasilkan banyak false positive; diskusikan distribusi.


### SEL 12 — 3D: explain Exchange/HashAggregate dan pengaruh shuffle partitions

In [14]:
q='SELECT wiki,COUNT(*) AS total,SUM(CASE WHEN bot=true THEN 1 ELSE 0 END) AS bot FROM wiki_events GROUP BY wiki'
old_shuffle=spark.conf.get('spark.sql.shuffle.partitions')
old_aqe=spark.conf.get('spark.sql.adaptive.enabled','true')
results=[]
try:
    # AQE nonaktif agar perbedaan shuffle partitions tampak eksplisit pada physical plan.
    spark.conf.set('spark.sql.adaptive.enabled','false')
    for n in (200,16):
        spark.conf.set('spark.sql.shuffle.partitions',str(n))
        print('\n========== EXPLAIN',n,'SHUFFLE PARTITIONS ==========')
        spark.sql(q).explain(True)
        sc.setJobGroup('S3_OPT_'+str(n),'Agregasi shuffle '+str(n))
        t0=time.perf_counter();rows=spark.sql(q).collect();secs=time.perf_counter()-t0
        results.append({'shuffle_partitions':n,'duration_seconds':round(secs,3),
                        'groups':len(rows),'total':sum(r['total'] for r in rows)})
        print('HASIL:',results[-1])
    assert results[0]['total']==results[1]['total']
finally:
    spark.conf.set('spark.sql.shuffle.partitions',old_shuffle)
    spark.conf.set('spark.sql.adaptive.enabled',old_aqe)
(HASIL/'s3_optimizer.json').write_text(json.dumps(results,indent=2),encoding='utf-8')
print('SIMPAN screenshot Spark UI: Exchange, HashAggregate partial/final, Job–Stage–Task.')



========== EXPLAIN 200 SHUFFLE PARTITIONS ==========


== Parsed Logical Plan ==
'Aggregate ['wiki], ['wiki, 'COUNT(1) AS total#1010, 'SUM(CASE WHEN ('bot = true) THEN 1 ELSE 0 END) AS bot#1011]
+- 'UnresolvedRelation [wiki_events], [], false

== Analyzed Logical Plan ==
wiki: string, total: bigint, bot: bigint
Aggregate [wiki#11], [wiki#11, count(1) AS total#1010L, sum(CASE WHEN (bot#16 = true) THEN 1 ELSE 0 END) AS bot#1011L]
+- SubqueryAlias wiki_events
   +- View (`wiki_events`, [id#9L, type#10, wiki#11, server_name#12, title#13, namespace#14, user#15, bot#16, minor#17, timestamp#18L, length#19, length_old#20L, length_new#21L, comment#22, event_time#23, old_byte#24L, new_byte#26L, selisih_byte#28L])
      +- Project [id#9L, type#10, wiki#11, server_name#12, title#13, namespace#14, user#15, bot#16, minor#17, timestamp#18L, length#19, length_old#20L, length_new#21L, comment#22, event_time#23, old_byte#24L, new_byte#26L, (new_byte#26L - old_byte#24L) AS selisih_byte#28L]
         +- Project [id#9L, type#10, wiki#11, server_name#12, title#

HASIL: {'shuffle_partitions': 200, 'duration_seconds': 203.498, 'groups': 344, 'total': 453974}

========== EXPLAIN 16 SHUFFLE PARTITIONS ==========


== Parsed Logical Plan ==
'Aggregate ['wiki], ['wiki, 'COUNT(1) AS total#1100, 'SUM(CASE WHEN ('bot = true) THEN 1 ELSE 0 END) AS bot#1101]
+- 'UnresolvedRelation [wiki_events], [], false

== Analyzed Logical Plan ==
wiki: string, total: bigint, bot: bigint
Aggregate [wiki#11], [wiki#11, count(1) AS total#1100L, sum(CASE WHEN (bot#16 = true) THEN 1 ELSE 0 END) AS bot#1101L]
+- SubqueryAlias wiki_events
   +- View (`wiki_events`, [id#9L, type#10, wiki#11, server_name#12, title#13, namespace#14, user#15, bot#16, minor#17, timestamp#18L, length#19, length_old#20L, length_new#21L, comment#22, event_time#23, old_byte#24L, new_byte#26L, selisih_byte#28L])
      +- Project [id#9L, type#10, wiki#11, server_name#12, title#13, namespace#14, user#15, bot#16, minor#17, timestamp#18L, length#19, length_old#20L, length_new#21L, comment#22, event_time#23, old_byte#24L, new_byte#26L, (new_byte#26L - old_byte#24L) AS selisih_byte#28L]
         +- Project [id#9L, type#10, wiki#11, server_name#12, title#

HASIL: {'shuffle_partitions': 16, 'duration_seconds': 214.725, 'groups': 344, 'total': 453974}
SIMPAN screenshot Spark UI: Exchange, HashAggregate partial/final, Job–Stage–Task.


In [15]:
spark.stop()
print("Spark Soal 3 sudah dihentikan")

Spark Soal 3 sudah dihentikan
